In [ ]:
import duckdb
path = "/home/math4ai/Documents/RCS/robot-control-stack/examples/teleop/box_pnp"
duckdb.sql(f"describe select *, unnest(obs.frames) from read_parquet('{path}')"), duckdb.sql(f"select count(*) as n_frames from read_parquet('{path}')")

In [ ]:
duckdb.sql(
f"""SELECT 
    info
FROM read_parquet('{path}')
WHERE success
limit 1
    """
)

In [ ]:
duckdb.sql(
    "SELECT count(DISTINCT uuid) as successful "
    f"FROM read_parquet('{path}') "
    "WHERE success"
),duckdb.sql(
    "SELECT count(DISTINCT uuid) as n_episodes "
    f"FROM read_parquet('{path}') "
)

In [ ]:
duckdb.sql(
f"""SELECT 
    MAX(step)/30, MIN(step)/30, AVG(step)/30
FROM read_parquet('{path}')
WHERE success
    """
)


In [ ]:
import io
from matplotlib import pyplot as plt
from PIL import Image
import duckdb
episode = 0
step = 100


uuids = duckdb.sql(f"SELECT DISTINCT uuid FROM read_parquet('{path}') WHERE success").fetchnumpy()
# uuids
uuid1 = uuids["uuid"][episode]
rel = duckdb.read_parquet(path)
success = rel.filter(f"uuid='{uuid1}' and step={step}").select("success").fetchone()[0]
info = rel.filter(f"uuid='{uuid1}' and step={step}").select("info").fetchone()[0]
count = rel.filter(f"uuid='{uuid1}'").count("*").fetchone()[0]
print(f"episode uuid: {uuid1}, success: {success}, n_steps: {count}, info: {info}")



# Create a figure with 1 row and 2 columns
fig, ax = plt.subplots(1, 3, figsize=(20, 5))


img1_data = rel.filter(f"uuid='{uuid1}' and step={step}").select("obs.frames.side.rgb.data").fetchone()[0]
image1 = Image.open(io.BytesIO(img1_data))
ax[0].imshow(image1)
ax[0].set_title("Head Camera")

img2_data = rel.filter(f"uuid='{uuid1}' and step={step}").select("obs.frames.digit_right_left.rgb.data").fetchone()[0]
image2 = Image.open(io.BytesIO(img2_data))
ax[1].imshow(image2)
ax[1].set_title("Left Wrist Camera") # Optional title

img3_data = rel.filter(f"uuid='{uuid1}' and step={step}").select("obs.frames.wrist.rgb.data").fetchone()[0]
image3 = Image.open(io.BytesIO(img3_data))
ax[2].imshow(image3)
ax[2].set_title("Right Wrist Camera") # Optional title



plt.show()

In [ ]:
import io
import duckdb
from PIL import Image as PILImage
from IPython.display import Image, display

uuids = duckdb.sql(
    f"SELECT DISTINCT uuid FROM read_parquet('{path}') WHERE success"
).fetchnumpy()
episode = 13
n_frames = 400
frame_stride = 9



uuid1 = uuids["uuid"][episode]
rel = duckdb.read_parquet(path)

count = rel.filter(f"uuid='{uuid1}'").count("*").fetchone()[0]
info = rel.filter(f"uuid='{uuid1}'").select("info").fetchone()[0]
success = rel.filter(f"uuid='{uuid1}'").select("success").fetchone()[0]

print(f"episode uuid: {uuid1}, success: {success}, n_steps: {count}, info: {info}")

steps = list(range(0, n_frames * frame_stride, frame_stride))

frames = (
    rel
    .filter(f"uuid='{uuid1}' and step in ({','.join(map(str, steps))})")
    .select("step, obs.frames.side.rgb.data AS img_data")
    .order("step")
    .fetchall()
)

print(f"Loaded {len(frames)} frames")

pil_frames = []

for step, img_data in frames:
    img = PILImage.open(io.BytesIO(img_data)).convert("RGB")
    pil_frames.append(img)

# Save to an in-memory GIF
gif_buffer = io.BytesIO()

pil_frames[0].save(
    gif_buffer,
    format="GIF",
    save_all=True,
    append_images=pil_frames[1:],
    duration=50,   # milliseconds per frame; 50 ms = 20 FPS
    loop=0,
)

gif_buffer.seek(0)

display(Image(data=gif_buffer.read(), format="gif"))

In [ ]:
import numpy as np
import pandas as pd
import duckdb
from matplotlib import pyplot as plt

episode = 13


uuid1 = uuids["uuid"][episode]

df = duckdb.sql(f"""
    SELECT
        step,
        obs.right.tquat AS obs_tquat,
        action.right.tquat AS action_tquat,
        obs.right.gripper AS obs_gripper,
        action.right.gripper AS action_gripper,
        info.right.absolute_action AS aa_tquat
    FROM read_parquet('{path}')
    WHERE uuid = '{uuid1}'
    ORDER BY step
""").df()

print(f"episode uuid: {uuid1}, n_steps: {len(df)}")

# Expand quaternion columns into x/y/z/w components
obs_tquat = np.stack(df["obs_tquat"][1:].to_numpy())
# print(df["action_tquat"].shape)
action_tquat = np.stack(df["action_tquat"][1:].to_numpy())
aa_tquat = np.stack(df["aa_tquat"][1:].to_numpy())
quat_names = ["x", "y", "z", "w"]
steps = df["step"][1:]

fig, ax = plt.subplots(1, 5, figsize=(26, 4), sharex=True)

for i, name in enumerate(quat_names):
    ax[i].plot(steps, obs_tquat[:, i], label="obs", linewidth=2)
    ax[i].plot(steps, action_tquat[:, i], label="action", linewidth=2, linestyle="--")
    ax[i].plot(steps, aa_tquat[:, i], label="AA", linewidth=2, linestyle="--")
    ax[i].set_title(f"right.tquat.{name}")
    ax[i].set_xlabel("step")
    ax[i].grid(True, alpha=0.3)
    ax[i].legend()

ax[4].plot(steps, df["obs_gripper"][1:], label="obs", linewidth=2)
ax[4].plot(steps, df["action_gripper"][1:], label="action", linewidth=2, linestyle="--")
ax[4].set_title("right.gripper")
ax[4].set_xlabel("step")
ax[4].grid(True, alpha=0.3)
ax[4].legend()

plt.suptitle(f"UUID: {uuid1}", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
duckdb.sql(
f"""SELECT 
    timestamp
FROM read_parquet('{path}')
    """
)

In [ ]:
duckdb.sql(
f"""WITH episode_stats AS (
    SELECT
        uuid,
        COUNT(*) AS frame_count,
        MAX("timestamp") - MIN("timestamp") AS duration_seconds
    FROM read_parquet('{path}')
    GROUP BY uuid
    HAVING duration_seconds > 0 -- Prevents division by zero for single-frame episodes
)
SELECT 
    AVG(frame_count / duration_seconds) AS average_frequency_hz
FROM episode_stats;
"""
)

In [ ]:
duckdb.sql(
f"""WITH episode_stats AS (
    SELECT
        uuid,
        COUNT(*) AS frame_count,
        MAX("timestamp") - MIN("timestamp") AS duration_seconds
    FROM read_parquet('{path}')
    GROUP BY uuid
    HAVING duration_seconds > 0 -- Prevents division by zero for single-frame episodes
)
SELECT 
    STDDEV(frame_count / duration_seconds) AS std_frequency_hz
FROM episode_stats;
"""
)

In [ ]:
duckdb.sql(
f"""WITH frame_intervals AS (
    SELECT 
        uuid,
        "timestamp",
        -- Get the timestamp of the *next* frame within the same episode
        LEAD("timestamp") OVER (PARTITION BY uuid ORDER BY "timestamp") AS next_timestamp
    FROM read_parquet('{path}')
),
instantaneous_frequencies AS (
    SELECT
        -- Frequency = 1 / time_difference
        1.0 / (next_timestamp - "timestamp") AS freq_hz
    FROM frame_intervals
    WHERE next_timestamp IS NOT NULL 
      AND (next_timestamp - "timestamp") > 1e-6 -- Filter out zero/tiny diffs to prevent errors
)
SELECT 
    AVG(freq_hz) AS mean_frequency_hz,
    STDDEV(freq_hz) AS std_frequency_hz
FROM instantaneous_frequencies;
"""
)

In [ ]:
duckdb.sql(
f"""WITH frame_intervals AS (
    SELECT 
        uuid,
        step,
        "timestamp",
        LEAD("timestamp") OVER (PARTITION BY uuid ORDER BY step) AS next_timestamp
    FROM read_parquet('{path}')
),
instantaneous_frequencies AS (
    SELECT
        uuid,
        step,
        1.0 / (next_timestamp - "timestamp") AS freq_hz
    FROM frame_intervals
    WHERE next_timestamp IS NOT NULL 
      AND (next_timestamp - "timestamp") > 1e-6
)
SELECT 
    uuid,
    -- Get the Min Frequency and the step it happened at
    MIN(freq_hz) AS min_freq_hz,
    ARG_MIN(step, freq_hz) AS step_at_min,
    
    -- Get the Max Frequency and the step it happened at
    MAX(freq_hz) AS max_freq_hz,
    ARG_MAX(step, freq_hz) AS step_at_max
FROM instantaneous_frequencies
GROUP BY uuid;
"""
)

# Save the <key> video from all episodes with stride as mp4

In [ ]:
import io
from pathlib import Path

import duckdb
import numpy as np
import imageio.v3 as iio
from PIL import Image as PILImage

# Config
output_dir = Path("episode_videos")
output_dir.mkdir(exist_ok=True)

camera_col = "obs.frames.side.rgb.data"
fps = 20
frame_stride = 10
n_frames = None  # set to e.g. 400 to cap each video, or None for full episode

con = duckdb.connect()

uuids = con.execute(
    f"""
    SELECT DISTINCT uuid
    FROM read_parquet('{path}')
    WHERE success
    ORDER BY uuid
    """
).fetchnumpy()["uuid"]

print(f"Found {len(uuids)} successful episodes")

for episode_idx, uuid1 in enumerate(uuids):
    limit_clause = f"LIMIT {n_frames}" if n_frames is not None else ""

    rows = con.execute(
        f"""
        SELECT
            step,
            {camera_col} AS img_data
        FROM read_parquet('{path}')
        WHERE uuid = ?
          AND step % ? = 0
        ORDER BY step
        {limit_clause}
        """,
        [str(uuid1), frame_stride],
    ).fetchall()

    if not rows:
        print(f"Skipping episode {episode_idx}: no frames found")
        continue

    frames = []
    target_size = None

    for step, img_data in rows:
        img = PILImage.open(io.BytesIO(img_data)).convert("RGB")

        # MP4 encoders expect all frames to have identical dimensions.
        if target_size is None:
            target_size = img.size
        elif img.size != target_size:
            img = img.resize(target_size)

        frames.append(np.asarray(img))

    video_path = output_dir / f"episode-{episode_idx}.mp4"

    iio.imwrite(
        video_path,
        frames,
        fps=fps,
        codec="libx264",
        pixelformat="yuv420p",
    )

    print(
        f"Saved {video_path} | uuid={uuid1} | frames={len(frames)} | "
        f"steps={rows[0][0]}..{rows[-1][0]}"
    )

print("Done.")